In [1]:
import numpy as np
import matplotlib.pyplot as plt
import csv
import os

In [ ]:
def generate_random_conic_type():
    """Возвращает случайный тип кривой: эллипс, парабола или гипербола."""
    return np.random.choice(["ellipse", "parabola", "hyperbola"])

def generate_noise_params():
    noise_std = np.random.uniform(7, 25)
    return noise_std

def generate_ellipse_data(noise_std, N=1000, x_range=(0,100)):
    # Выбираем параметры эллипса так, чтобы он был в пределах [0,100] по x
    cx = np.random.uniform(30, 70)
    cy = np.random.uniform(30, 70)
    a_axis = np.random.uniform(10, 30)
    b_axis = np.random.uniform(10, 30)
    theta = np.random.uniform(0, np.pi)
    
    t = np.linspace(0, 2*np.pi, N)
    x_clean = cx + a_axis * np.cos(t)*np.cos(theta) - b_axis * np.sin(t)*np.sin(theta)
    y_clean = cy + a_axis * np.cos(t)*np.sin(theta) + b_axis * np.sin(t)*np.cos(theta)
    
    # Добавляем независимый аддитивный шум к x и y
    x_noisy = x_clean + np.random.normal(0, noise_std, N)
    y_noisy = y_clean + np.random.normal(0, noise_std, N)
    
    # Вычисляем коэффициенты общего уравнения эллипса.
    # Стандартная форма: ((x-cx)*cosθ+(y-cy)*sinθ)^2/a_axis^2 + (-(x-cx)*sinθ+(y-cy)*cosθ)^2/b_axis^2 = 1.
    A_coef = (np.cos(theta)**2)/(a_axis**2) + (np.sin(theta)**2)/(b_axis**2)
    B_coef = 2*np.cos(theta)*np.sin(theta)*(1/(a_axis**2) - 1/(b_axis**2))
    C_coef = (np.sin(theta)**2)/(a_axis**2) + (np.cos(theta)**2)/(b_axis**2)
    D_coef = -2*A_coef*cx - B_coef*cy
    E_coef = -B_coef*cx - 2*C_coef*cy
    F_coef = A_coef*cx**2 + B_coef*cx*cy + C_coef*cy**2 - 1
    coeffs = (A_coef, B_coef, C_coef, D_coef, E_coef, F_coef)
    return x_clean, y_clean, x_noisy, y_noisy, coeffs

def generate_parabola_data(noise_std, N=1000, x_range=(0,100)):
    # Выбираем параметры параболы вида y = A*(x-h)^2 + k
    h = np.random.uniform(30, 70)
    k = np.random.uniform(0, 100)
    A = np.random.uniform(-0.1, 0.1)
    
    x_clean = np.linspace(x_range[0], x_range[1], N)
    y_clean = A*(x_clean - h)**2 + k
    x_noisy = x_clean + np.random.normal(0, noise_std, N)
    y_noisy = y_clean + np.random.normal(0, noise_std, N)
    
    # Приводим уравнение к виду: A*x^2 - 2A*h*x + A*h^2 + y - k = 0.
    A_coef = A
    B_coef = 0
    C_coef = 0
    D_coef = -2*A*h
    E_coef = 1
    F_coef = A*h**2 - k
    coeffs = (A_coef, B_coef, C_coef, D_coef, E_coef, F_coef)
    return x_clean, y_clean, x_noisy, y_noisy, coeffs

def generate_hyperbola_data(noise_std, N=1000, x_range=(0,100)):
    # Выбираем параметры горизонтальной гиперболы: (x-h)^2/a^2 - (y-k)^2/b^2 = 1, для правой ветви.
    h = np.random.uniform(30, 70)
    k = np.random.uniform(30, 70)
    a = np.random.uniform(10, 20)
    b = np.random.uniform(10, 20)
    
    # Для правой ветви x от (h+a) до x_range[1]
    x_vals = np.linspace(h + a, x_range[1], N)
    y_clean = k + b * np.sqrt((x_vals - h)**2 / a**2 - 1)
    x_noisy = x_vals + np.random.normal(0, noise_std, N)
    y_noisy = y_clean + np.random.normal(0, noise_std, N)
    
    # Перепишем уравнение гиперболы: (x-h)^2/a^2 - (y-k)^2/b^2 = 1  =>
    # (1/a^2)x^2 - (2h/a^2)x + (h^2/a^2) - (1/b^2)y^2 + (2k/b^2)y - (k^2/b^2) - 1 = 0.
    A_coef = 1/a**2
    B_coef = 0
    C_coef = -1/b**2
    D_coef = -2*h/a**2
    E_coef = 2*k/b**2
    F_coef = (h**2)/(a**2) - (k**2)/(b**2) - 1
    coeffs = (A_coef, B_coef, C_coef, D_coef, E_coef, F_coef)
    return x_vals, y_clean, x_noisy, y_noisy, coeffs

def save_data_to_csv(X, Y, filename):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['x', 'y'])
        for xi, yi in zip(X, Y):
            writer.writerow([xi, yi])


def generate_outliers(num_outliers, center_x, center_y, std):
    x_out = np.random.normal(center_x, std, num_outliers)
    y_out = np.random.normal(center_y, std, num_outliers)
    return x_out, y_out

In [3]:
def gen_conic_outlier(n_out, k):
    # Выбираем случайный тип квадратичной кривой
    conic_type = generate_random_conic_type()
    noise_std = generate_noise_params()
    N = 1000
    x_range = (0, 100)
    
    if conic_type == "ellipse":
        x_clean, y_clean, x_noisy, y_noisy, coeffs = generate_ellipse_data(noise_std, N, x_range)
    elif conic_type == "parabola":
        x_clean, y_clean, x_noisy, y_noisy, coeffs = generate_parabola_data(noise_std, N, x_range)
    else:  # hyperbola
        x_clean, y_clean, x_noisy, y_noisy, coeffs = generate_hyperbola_data(noise_std, N, x_range)
    
    # Аккумулируем выбросы
    outlier_x_list = []
    outlier_y_list = []
    filename_base = f"{conic_type}_{coeffs[0]:.2f}_{coeffs[1]:.2f}_{coeffs[2]:.2f}_{coeffs[3]:.2f}_{coeffs[4]:.2f}_{coeffs[5]:.2f}"
    
    for _ in range(n_out):
        # Для выбросов выбираем случайное значение x и ищем ближайшую точку чистой кривой
        rand_x = np.random.uniform(x_range[0], x_range[1])
        idx = np.argmin(np.abs(x_clean - rand_x))
        true_y = y_clean[idx]
        # Смещаем точку по y на случайное значение (может быть как вверх, так и вниз)
        outlier_center_y = true_y + np.random.uniform(20, 50) * np.random.choice([-1, 1])
        outlier_center_x = rand_x
        outlier_std = np.random.uniform(2, 7)
        num_outliers = int((50*k)/(1-0.05*k))  # аналогичная формула с параметром k
        x_outliers, y_outliers = generate_outliers(num_outliers, outlier_center_x, outlier_center_y, outlier_std)
        outlier_x_list.append(x_outliers)
        outlier_y_list.append(y_outliers)
        filename_base += f"_{num_outliers}"
        
    # Объединяем данные: шумная кривая + выбросы
    X = np.concatenate((x_noisy, *outlier_x_list))
    Y = np.concatenate((y_noisy, *outlier_y_list))
    
    csv_filename = f"data_outliers/{filename_base}.csv"
    plot_filename = f"graph_outliers/{filename_base}.png"
    
    save_data_to_csv(X, Y, csv_filename)
    
    plt.figure(figsize=(8, 6))
    plt.scatter(X, Y, color='gray', alpha=0.7, label='Данные с выбросами')
    plt.plot(x_clean, y_clean, color='red', linewidth=2, label='Чистая кривая')
    plt.title(f'{conic_type.capitalize()} с аддитивным шумом и выбросами')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.grid(True)
    
    plt.savefig(plot_filename)
    print(f"График успешно сохранён в файл '{plot_filename}'")
    plt.show()

# for i in range(1, 6):
#     for k in range(14):
#         gen_conic_outlier(i, k)